# GRUEncoder Training — SOLUSDT Phase A

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sandeep999-cyber/solusdt-ml-trader/blob/main/colab_train.ipynb)

### Workflow (no uploads needed)
1. **Open** via the badge above — Colab loads the latest version directly from GitHub
2. **Code changes** → `git push`; re-open from the badge to get the updated notebook
3. **Data** (upload once to Drive): `MyDrive/ModelProject/data_processed/`
4. **Checkpoints** auto-saved to Drive; sync locally via `scripts/pull_checkpoint.py`

In [ ]:
# Cell 0: Mount Google Drive (stores data + checkpoints)
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted at /content/drive')

In [ ]:
# Cell 1: Pull latest code from GitHub, symlink data from Drive
import os, sys, json
from pathlib import Path

REPO_URL = 'https://github.com/sandeep999-cyber/solusdt-ml-trader.git'
PROJECT_DIR = '/content/ModelProject'
NOTEBOOK_PATH = os.path.join(PROJECT_DIR, 'colab_train.ipynb')

# Clone/pull the repo
if os.path.exists(PROJECT_DIR):
    %cd "$PROJECT_DIR"
    !git pull
    print('Pulled latest code from GitHub')
else:
    !git clone "$REPO_URL" "$PROJECT_DIR"
    %cd "$PROJECT_DIR"
    print('Cloned repository from GitHub')

# Persist the notebook to Drive as a convenience backup
DRIVE_NB_DIR = '/content/drive/MyDrive/ModelProject'
os.makedirs(DRIVE_NB_DIR, exist_ok=True)
DRIVE_NB = os.path.join(DRIVE_NB_DIR, 'colab_train.ipynb')
import shutil
shutil.copy(NOTEBOOK_PATH, DRIVE_NB)
print(f'Notebook synced to Drive: {DRIVE_NB}')

# Symlink data from Drive
DRIVE_DATA = '/content/drive/MyDrive/ModelProject/data_processed'
LOCAL_DATA = 'data/processed/v1/SOLUSDT/1m'
if not os.path.exists(LOCAL_DATA):
    if not os.path.exists(DRIVE_DATA):
        raise FileNotFoundError(
            f'Processed data not found on Drive at {DRIVE_DATA}. '
            'Upload data/processed/ to MyDrive/ModelProject/data_processed/ first.'
        )
    os.makedirs(os.path.dirname(LOCAL_DATA), exist_ok=True)
    try:
        os.symlink(DRIVE_DATA, LOCAL_DATA)
        print(f'Symlinked data from Drive: {DRIVE_DATA} -> {LOCAL_DATA}')
    except AttributeError:
        shutil.copytree(DRIVE_DATA, LOCAL_DATA, dirs_exist_ok=True)
        print(f'Copied data from Drive to {LOCAL_DATA}')
else:
    print(f'Data already available at {LOCAL_DATA}')

print(f'\nProject ready at {PROJECT_DIR}')
print(f'Contents: {os.listdir(".")}')

In [ ]:
# Cell 2: Verify GPU
import torch
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
else:
    raise RuntimeError('GPU not available — go to Runtime > Change runtime type > T4 GPU')

In [ ]:
# Cell 3: Install Python dependencies
!pip install --quiet pandas pyarrow pyyaml httpx numba
print('Dependencies installed')

In [ ]:
# Cell 4: Smoke test only (~10s)
# Run a 2-epoch smoke test to verify everything works before the long run.
# This does NOT run full training — that's Cell 5.
smoke_ok = False
try:
    !python -m model.train --config configs/phase_a_gru_h32.yaml --smoke-test-only
    smoke_ok = True
    print('\nSmoke test PASSED — ready for full training.')
except Exception as e:
    print(f'Smoke test FAILED: {e}')

In [ ]:
# Cell 5: Full training run (GRU h32)
# Takes ~3-5 minutes on a T4 GPU (vs ~2h on CPU)
if smoke_ok:
    !python -m model.train --config configs/phase_a_gru_h32.yaml
else:
    print('Skipping full training — smoke test did not pass. Fix errors first.')

---
### Diagnostic: fixed-variance GRU (is the flat mean a training artifact?)

Forces `log_var=0` so loss = plain MSE. If MSE drops below ~1.016 (the
unconditional variance that every NLL-trained run plateaus at), the flat
line was a variance-shortcut pathology, not a genuine predictive ceiling.

Run Cells 6a→6b after Cell 5 completes. Results go to a separate run dir.

In [ ]:
# Cell 6a: Diagnostic smoke test
diag_smoke_ok = False
try:
    !python -m model.train --config configs/diag_fixed_var.yaml --smoke-test-only
    diag_smoke_ok = True
    print('\nDiagnostic smoke test PASSED.')
except Exception as e:
    print(f'Diagnostic smoke test FAILED: {e}')

In [ ]:
# Cell 6b: Diagnostic full run (15 epochs)
if diag_smoke_ok:
    !python -m model.train --config configs/diag_fixed_var.yaml
else:
    print('Skipping diagnostic — smoke test did not pass.')

In [ ]:
# Cell 7: Save checkpoints to Drive
import json
from pathlib import Path

RUNS_DIR = Path('model/runs')
run_dirs = sorted(RUNS_DIR.iterdir())
if not run_dirs:
    raise RuntimeError('No training runs found')

real_runs = [d for d in run_dirs if d.is_dir() and 'smoketest' not in d.name]
if not real_runs:
    real_runs = [d for d in run_dirs if d.is_dir()]
latest_run = real_runs[-1].name
print(f'Latest run: {latest_run}')

# --- Save all runs to Drive ---
DRIVE_RUN = Path('/content/drive/MyDrive/ModelProject/checkpoints')
DRIVE_RUN.mkdir(parents=True, exist_ok=True)
import shutil

def save_run(run_name):
    best_path = RUNS_DIR / run_name / 'checkpoints' / 'best.pt'
    if not best_path.exists():
        print(f'  No best.pt for {run_name}')
        return
    shutil.copy(best_path, DRIVE_RUN / f'{run_name}_best.pt')
    print(f'  Saved {run_name}/best.pt')

    config_path = RUNS_DIR / run_name / 'config.yaml'
    if config_path.exists():
        shutil.copy(config_path, DRIVE_RUN / f'{run_name}_config.yaml')

    metrics_path = RUNS_DIR / run_name / 'metrics.jsonl'
    if metrics_path.exists():
        shutil.copy(metrics_path, DRIVE_RUN / f'{run_name}_metrics.jsonl')
        print(f'  Saved {run_name} metrics')
        with open(metrics_path) as f:
            lines = f.readlines()
        last = json.loads(lines[-1])
        print(f'    Final epoch: {last.get("epoch", "?")}  MSE: {last.get("mse", "?")}  loss: {last.get("loss", "?")}')

for run_dir in real_runs:
    save_run(run_dir.name)

# Update latest pointer for local sync
best_path = RUNS_DIR / latest_run / 'checkpoints' / 'best.pt'
if best_path.exists():
    shutil.copy(best_path, DRIVE_RUN / 'best.pt')
    print(f'\nCopied {latest_run}/best.pt -> best.pt (latest pointer)')
    from datetime import datetime, timezone
    pointer = {
        'run_name': latest_run,
        'checkpoint_path': str(DRIVE_RUN / f'{latest_run}_best.pt'),
        'updated': datetime.now(timezone.utc).isoformat(),
    }
    with open(DRIVE_RUN / 'latest.json', 'w') as f:
        json.dump(pointer, f, indent=2)
    print(f'Saved pointer -> latest.json')
else:
    print('No best.pt found for latest run')

In [ ]:
# Cell 8: Download a checkpoint (optional — for manual download)
from google.colab import files

RUNS_DIR = Path('model/runs')
run_dirs = sorted(RUNS_DIR.iterdir())
real_runs = [d for d in run_dirs if d.is_dir() and 'smoketest' not in d.name]
latest_run = real_runs[-1].name if real_runs else (run_dirs[-1].name if run_dirs else None)

if latest_run:
    best_path = RUNS_DIR / latest_run / 'checkpoints' / 'best.pt'
    if best_path.exists():
        files.download(str(best_path))
        print(f'Downloading {latest_run}/best.pt...')
    else:
        print(f'No best.pt in {latest_run}')
else:
    print('No runs found')